# 🚁 Lab 1: Multirotor Vehicle Design, BEMT & Propulsion Sizing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blaze505050/drone-digital-twin/blob/main/examples/labs/Lab1_Vehicle_Design_and_BEMT.ipynb)

Welcome to **Lab 1 of the DronePy Aerospace & Robotics Curriculum**!
In this lab, you will explore multirotor propulsion physics, compare simple momentum theory with **Blade Element Momentum Theory (BEMT)**, and size brushless DC motors and propellers for optimal thrust-to-weight ratio and flight endurance.

---
### 🎯 Learning Objectives
1. Understand the physics of rotary-wing thrust generation and induced velocity.
2. Formulate thrust coefficient $C_T$, power coefficient $C_P$, and Figure of Merit (FM).
3. Evaluate 10-inch vs 12-inch propeller performance using DronePy's aerodynamic models.
4. Calculate hover equilibrium RPM, electrical power consumption, and battery endurance.


In [ ]:
# Setup Google Colab dependencies if running in the cloud
import sys
try:
    import dronepy
except ImportError:
    !pip install -q git+https://github.com/blaze505050/drone-digital-twin.git
    import dronepy

import numpy as np
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

print(f"DronePy Version: {dronepy.__version__}")


---
## 1. Mathematical Theory: Momentum Theory vs BEMT

### A. Simple Actuator Disk (Rankine-Froude) Momentum Theory
In ideal momentum theory, the propeller is treated as an infinitesimally thin actuator disk of area $A = \pi R^2$.
Applying conservation of mass, momentum, and energy across the disk in hover yields the induced velocity $v_i$:

$$v_i = \sqrt{\frac{T}{2 \rho A}}$$

The ideal induced power required to hover is:
$$P_i = T \cdot v_i = \frac{T^{3/2}}{\sqrt{2 \rho A}}$$

### B. Blade Element Momentum Theory (BEMT)
Simple momentum theory ignores blade profile drag, tip vortices, radial chord variations, and blade twist. **BEMT** discretizes the blade into $N$ radial elements of width $dr$. For each element at radius $r$:

$$dL = \frac{1}{2} \rho V_e^2 c(r) C_l(\alpha) dr$$
$$dD = \frac{1}{2} \rho V_e^2 c(r) C_d(\alpha) dr$$

Where:
- $V_e = \sqrt{V_a^2 + (\Omega r - v_i)^2}$ is the local effective airspeed
- $\alpha = \theta(r) - \arctan(V_a / (\Omega r - v_i))$ is the effective angle of attack
- $F(r) = \frac{2}{\pi} \arccos\left(\exp\left(-\frac{B(R - r)}{2 r \sin\phi}\right)\right)$ is Prandtl's tip-loss factor

The non-dimensional aerodynamic coefficients are defined as:
$$C_T = \frac{T}{\rho n^2 D^4}, \quad C_P = \frac{P}{\rho n^3 D^5}, \quad \text{Figure of Merit (FM)} = \frac{P_i}{P_{\text{actual}}} = \frac{C_T^{3/2}}{\sqrt{2} C_P}$$
where $n = \text{RPM} / 60$ (rev/s) and $D$ is the rotor diameter in meters.


In [ ]:
# 2. Evaluating Propeller Aerodynamics with DronePy
# Comparing parametric wind-tunnel calibrated propeller with BEMT solver
prop_std = dronepy.Propeller(diameter_in=10.0, pitch_in=4.5)
prop_bemt = dronepy.BEMTPropeller(diameter_in=10.0, pitch_in=4.5)

test_rpms = np.linspace(2000, 9000, 8)
print(f"{'RPM':>6} | {'Std Thrust (N)':>15} | {'BEMT Thrust (N)':>15} | {'Std Power (W)':>15}")
print("-" * 60)

for rpm in test_rpms:
    t_std, q_std, p_std = prop_std.compute_thrust_and_torque(rpm)
    t_bemt, q_bemt, p_bemt = prop_bemt.compute_thrust_and_torque(rpm)
    print(f"{rpm:6.0f} | {t_std:15.3f} | {t_bemt:15.3f} | {p_std:15.2f}")


---
## 3. Vehicle Assembly & Hover Equilibrium
A standard quadcopter has 4 rotors. In hover equilibrium at sea level:
$$T_{\text{total}} = m \cdot g \implies T_{\text{rotor}} = \frac{m \cdot g}{4}$$

Let us construct a 1.80 kg quadcopter in DronePy and calculate its equilibrium hover point.


In [ ]:
# Create a 1.8 kg quadcopter with 10x4.5 propellers
mass_kg = 1.80
g = 9.80665
hover_thrust_total = mass_kg * g
hover_thrust_per_rotor = hover_thrust_total / 4.0

prop = dronepy.Propeller(diameter_in=10.0, pitch_in=4.5)

# Find hover RPM by inverting thrust curve
def find_hover_rpm(propeller, target_thrust):
    rpms = np.linspace(2000, 10000, 1000)
    thrusts = [propeller.compute_thrust_and_torque(r)[0] for r in rpms]
    return float(np.interp(target_thrust, thrusts, rpms))

hover_rpm = find_hover_rpm(prop, hover_thrust_per_rotor)
_, _, hover_mech_power_per_rotor = prop.compute_thrust_and_torque(hover_rpm)
total_hover_power_w = hover_mech_power_per_rotor * 4.0

print(f"Total Vehicle Weight:     {hover_thrust_total:.2f} N")
print(f"Hover Thrust per Rotor:   {hover_thrust_per_rotor:.2f} N")
print(f"Hover Motor Speed:        {hover_rpm:.1f} RPM")
print(f"Total Mechanical Power:   {total_hover_power_w:.1f} W")


---
## 📝 Student Exercise: Search & Rescue Sizing Challenge

### Mission Briefing:
You are designing a **Search & Rescue (SAR)** quadcopter with an All-Up Weight (AUW) of **$m = 2.40\text{ kg}$**.
You have two propulsion candidates:
- **Option A (High-RPM/Small-Prop)**: $10 \times 4.5$ propellers, Max RPM = 9,500.
- **Option B (Low-RPM/Large-Prop)**: $12 \times 4.5$ propellers, Max RPM = 7,500.

### Your Tasks:
1. Compute the hover RPM required per rotor for both Option A and Option B.
2. Determine the **Thrust-to-Weight Ratio (TWR)** at maximum RPM for both options:
   $$\text{TWR} = \frac{4 \cdot T_{\text{max}}}{m \cdot g}$$
   *(Aerospace standard requires $\text{TWR} \ge 2.0$ for gust authority).*
3. Calculate the total mechanical hover power for both options and determine which option yields greater endurance on a 4S LiPo battery (14.8 V nominal, 5000 mAh capacity, 80% usable depth-of-discharge = 59.2 Wh).


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STUDENT SOLUTION CELL - Complete the calculations below:
# ══════════════════════════════════════════════════════════════════
sar_mass_kg = 2.40
sar_hover_thrust_per_rotor = (sar_mass_kg * 9.80665) / 4.0
battery_energy_wh = 14.8 * 5.0 * 0.80  # 59.2 Wh

prop_a = dronepy.Propeller(diameter_in=10.0, pitch_in=4.5)
prop_b = dronepy.Propeller(diameter_in=12.0, pitch_in=4.5)

# 1. Calculate hover RPM
hover_rpm_a = find_hover_rpm(prop_a, sar_hover_thrust_per_rotor)
hover_rpm_b = find_hover_rpm(prop_b, sar_hover_thrust_per_rotor)

# 2. Calculate max thrust and TWR
max_t_a, _, _ = prop_a.compute_thrust_and_torque(9500.0)
max_t_b, _, _ = prop_b.compute_thrust_and_torque(7500.0)
twr_a = (4 * max_t_a) / (sar_mass_kg * 9.80665)
twr_b = (4 * max_t_b) / (sar_mass_kg * 9.80665)

# 3. Calculate power and endurance (assuming 85% electrical-to-mechanical efficiency)
_, _, p_a = prop_a.compute_thrust_and_torque(hover_rpm_a)
_, _, p_b = prop_b.compute_thrust_and_torque(hover_rpm_b)
total_p_a = (4 * p_a) / 0.85
total_p_b = (4 * p_b) / 0.85

endurance_min_a = (battery_energy_wh / total_p_a) * 60.0
endurance_min_b = (battery_energy_wh / total_p_b) * 60.0

print(f"Option A (10x4.5): Hover {hover_rpm_a:.0f} RPM | TWR: {twr_a:.2f} | Pwr: {total_p_a:.1f} W | Endurance: {endurance_min_a:.1f} min")
print(f"Option B (12x4.5): Hover {hover_rpm_b:.0f} RPM | TWR: {twr_b:.2f} | Pwr: {total_p_b:.1f} W | Endurance: {endurance_min_b:.1f} min")

# Validation Assertions
assert twr_a >= 2.0, "TWR A must exceed 2.0"
assert twr_b >= 2.0, "TWR B must exceed 2.0"
assert endurance_min_b > endurance_min_a, "Larger prop should yield longer endurance due to lower disc loading"
print("SUCCESS: Lab 1 sizing calculations verified successfully!")
